# 🚀 eKYC CCCD Detection — SSD MobileNet V2 FPNLite 320×320

Pipeline huấn luyện và xuất mô hình nhận diện thẻ CCCD Việt Nam sang **TensorFlow Lite**.

## Kiến trúc pipeline
```
Dataset YOLO  →  TFRecord  →  Fine-tune SSD MobileNetV2  →  SavedModel  →  TFLite
```

## Lớp nhận diện
| TF ID | Tên | Ý nghĩa |
|-------|-----|---------|
| 1 | bottom_left | Góc dưới trái thẻ |
| 2 | bottom_right | Góc dưới phải thẻ |
| 3 | image_person | Vùng ảnh chân dung |
| 4 | top_left | Góc trên trái thẻ |
| 5 | top_right | Góc trên phải thẻ |

## Lý do chọn SSD MobileNet V2 FPNLite
- Google thiết kế **riêng cho TFLite** — latency ~22ms trên Pixel 4 CPU
- **FPN** (Feature Pyramid Network) phát hiện tốt các góc nhỏ trên thẻ
- Checkpoint COCO pretrained giúp fine-tune hội tụ nhanh trên 462 ảnh
- EfficientDet chính xác hơn nhưng nặng gấp 3× — không cần thiết cho 5 lớp đơn giản

> ⏱️ **Thời gian ước tính:** ~1–2 giờ GPU T4 (Colab Free)

In [ ]:
# Kiểm tra GPU — bắt buộc phải có GPU để train đủ nhanh
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU khả dụng:')
    print(result.stdout[:600])
else:
    print('⚠️  KHÔNG TÌM THẤY GPU!')
    print('→ Vào Runtime > Change runtime type > Hardware accelerator > GPU (T4)')

---
## 📦 Bước 1: Cài đặt thư viện

Cài **TensorFlow Object Detection API** (`tf-models-official`) và các gói hỗ trợ.

> ⏳ Quá trình này mất ~5 phút — chạy một lần duy nhất mỗi session.

In [ ]:
%%capture install_log
# ============================================================
#  CÀI ĐẶT CHO COLAB 2025 (Python 3.12 + TF 2.16+ + Keras 3)
# ============================================================

# 1. protobuf compiler
!apt-get install -q protobuf-compiler

# 2. Clone TF Models (bỏ qua nếu đã tồn tại)
![ -d /content/tf_models ] || git clone --depth 1 https://github.com/tensorflow/models /content/tf_models

# 3. Compile .proto → *_pb2.py
!cd /content/tf_models/research && protoc object_detection/protos/*.proto --python_out=.

# 4. Cài object_detection từ source
!pip install -q /content/tf_models/research/

# 5. tf-keras: khôi phục Keras 2 API (fix "layers has no attribute experimental")
!pip install -q tf-keras

# 6. Các dependencies hay thiếu trên Colab
!pip install -q lvis scipy shapely pycocotools

# 7. Thư viện xử lý ảnh
!pip install -q Pillow tqdm opencv-python-headless matplotlib

In [ ]:
import os

BASE = '/content/tf_models/research/object_detection'

def patch_optional_import(path, import_fragment, alias, label):
    """
    Bọc một dòng 'import X as Y' thành try/except, tự động giữ đúng indentation.

    Vấn đề với str.replace() đơn giản:
        Original (inside an if/try block với indent 2 spaces):
            if condition:
              import tensorflow_io as tfio
        Sau khi replace không tính indent:
            if condition:
              try:
                import tensorflow_io as tfio
            except ImportError:          ← indent sai → SyntaxError
                tfio = None

    Cách fix: đọc từng dòng, phát hiện indent của dòng gốc,
    rồi xây try/except với cùng indent đó.
    """
    if not os.path.isfile(path):
        print(f'  SKIP (không tìm thấy): {label}')
        return

    with open(path) as f:
        lines = f.readlines()

    # Nếu đã patch rồi thì bỏ qua
    if any(f'{alias} = None' in ln for ln in lines):
        print(f'  Already patched: {label}')
        return

    new_lines = []
    changed = False
    for line in lines:
        if import_fragment in line and line.lstrip().startswith('import '):
            indent = len(line) - len(line.lstrip())   # số spaces gốc
            pad    = ' ' * indent                      # indent của try/except
            inner  = ' ' * (indent + 4)               # indent của body bên trong
            stripped = line.lstrip()                   # dòng gốc (giữ comment)
            new_lines.append(f'{pad}try:\n')
            new_lines.append(f'{inner}{stripped}')     # giữ nguyên dòng import + comment
            new_lines.append(f'{pad}except ImportError:\n')
            new_lines.append(f'{inner}{alias} = None  # {label}\n')
            changed = True
        else:
            new_lines.append(line)

    if changed:
        with open(path, 'w') as f:
            f.writelines(new_lines)
        print(f'  Patched: {label}')
    else:
        print(f'  Pattern not found (có thể đã thay đổi): {label}')


# --- Áp dụng patch ---

# deepmac_meta_arch.py dùng tensorflow_io (không có trên TF 2.17+)
# SSD không dùng DeepMAC, nhưng model_builder.py import nó vô điều kiện
patch_optional_import(
    path            = f'{BASE}/meta_architectures/deepmac_meta_arch.py',
    import_fragment = 'import tensorflow_io as tfio',
    alias           = 'tfio',
    label           = 'deepmac_meta_arch — không cần cho SSD',
)

print('\nPatch xong — tiếp tục chạy cell verify bên dưới')

In [ ]:
import os, sys

# PHẢI set trước khi import tensorflow — sau khi import thì không có tác dụng
os.environ['TF_USE_LEGACY_KERAS']                     = '1'   # dùng Keras 2 API thay Keras 3
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION']  = 'python'  # tránh _pb2.py TypeError

# Thêm TF Models vào Python path (cần sau mỗi restart runtime)
for p in ['/content/tf_models/research', '/content/tf_models/research/slim']:
    if p not in sys.path:
        sys.path.insert(0, p)

import tensorflow as tf
print(f'TensorFlow : {tf.__version__}')
print(f'Python     : {sys.version.split()[0]}')
print(f'GPU        : {tf.config.list_physical_devices("GPU")}')
print(f'Keras legacy mode : {os.environ["TF_USE_LEGACY_KERAS"]}')

try:
    import object_detection
    import importlib.util
    spec = importlib.util.find_spec('object_detection')
    print(f'object_detection API : OK  ({spec.origin})')
except ImportError:
    print('FAILED — restart runtime va chay lai cell cai dat')

---
## ⚙️ Bước 2: Cấu hình toàn cục

**Chỉnh các giá trị trong cell này trước khi chạy toàn bộ notebook.**
- `NUM_STEPS` — số bước train (10 000 bước ≈ 1.5 giờ GPU T4)
- `BATCH_SIZE` — giảm xuống 4 nếu bị lỗi OOM
- `SAVE_TO_DRIVE` — `True` để tự động lưu kết quả vào Google Drive

In [ ]:
import os

# ============================================================
#  CẤU HÌNH — chỉnh tại đây
# ============================================================

CLASSES     = ['bottom_left', 'bottom_right', 'image_person', 'top_left', 'top_right']
NUM_CLASSES = len(CLASSES)

IMAGE_SIZE   = 320
BATCH_SIZE   = 8        # giảm xuống 4 nếu lỗi OOM
NUM_STEPS    = 10_000
WARMUP_STEPS = 1_000
LR_BASE      = 0.08
WARMUP_LR    = 0.013333

SAVE_TO_DRIVE = True

# ============================================================
#  ĐƯỜNG DẪN
# ============================================================

WORK_DIR        = '/content/ekyc_tf'
DATA_DIR        = f'{WORK_DIR}/data'
TFRECORD_DIR    = f'{DATA_DIR}/tfrecords'
LABEL_MAP_PATH  = f'{DATA_DIR}/label_map.pbtxt'
PRETRAINED_DIR  = f'{WORK_DIR}/pretrained'
CONFIG_PATH     = f'{WORK_DIR}/configs/pipeline.config'
CKPT_DIR        = f'{WORK_DIR}/checkpoints'
EXPORT_DIR      = f'{WORK_DIR}/exported_model'
TFLITE_DIR      = f'{WORK_DIR}/tflite'
DRIVE_SAVE_DIR  = '/content/drive/MyDrive/ekyc_cccd'

MODEL_NAME      = 'ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8'
PRETRAINED_CKPT = f'{PRETRAINED_DIR}/{MODEL_NAME}/checkpoint/ckpt-0'

for d in [DATA_DIR, TFRECORD_DIR,
          f'{DATA_DIR}/images/train', f'{DATA_DIR}/images/val',
          f'{DATA_DIR}/labels/train', f'{DATA_DIR}/labels/val',
          PRETRAINED_DIR, f'{WORK_DIR}/configs',
          CKPT_DIR, EXPORT_DIR, TFLITE_DIR]:
    os.makedirs(d, exist_ok=True)

print('Cau hinh OK')
print(f'  Classes  : {CLASSES}')
print(f'  WORK_DIR : {WORK_DIR}')

---
## 💾 Bước 3: Mount Google Drive & Upload Dataset

Dataset YOLO cần có cấu trúc:
```
data/
├── images/
│   ├── train/   ← ảnh train (.jpg / .png)
│   └── val/     ← ảnh validation
└── labels/
    ├── train/   ← nhãn YOLO (.txt)
    └── val/
```
Mỗi dòng trong file `.txt`: `class_id cx cy width height` (tọa độ 0–1)

**Chọn một trong hai cách bên dưới:**

In [ ]:
# Mount Google Drive — bỏ qua nếu không dùng Drive
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive đã được mount')

In [ ]:
# === CÁCH A: Upload file ZIP từ máy tính ===
# Nén thư mục dataset thành ZIP rồi upload lên đây
# Cấu trúc ZIP: dataset.zip/images/train/*.jpg  dataset.zip/labels/train/*.txt ...

import zipfile
from google.colab import files

print('Chọn file ZIP chứa dataset (images/ + labels/)...')
uploaded = files.upload()

for fname in uploaded:
    zip_path = f'/tmp/{fname}'
    with open(zip_path, 'wb') as f:
        f.write(uploaded[fname])
    print(f'Đang giải nén {fname}...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(DATA_DIR)
    os.remove(zip_path)
    print(f'✅ Giải nén xong → {DATA_DIR}')

In [ ]:
# === CÁCH B: Copy từ Google Drive ===
# Điều chỉnh DRIVE_DATASET cho khớp với thư mục của bạn trên Drive

import shutil
DRIVE_DATASET = f'{DRIVE_SAVE_DIR}/dataset'

if os.path.isdir(DRIVE_DATASET):
    shutil.copytree(f'{DRIVE_DATASET}/images', f'{DATA_DIR}/images', dirs_exist_ok=True)
    shutil.copytree(f'{DRIVE_DATASET}/labels', f'{DATA_DIR}/labels', dirs_exist_ok=True)
    print(f'✅ Đã copy dataset từ {DRIVE_DATASET}')
else:
    print(f'⚠️  Không tìm thấy {DRIVE_DATASET} — hãy dùng Cách A hoặc upload thủ công')

In [ ]:
# Kiểm tra dataset — phải thấy ảnh và nhãn trước khi tiếp tục
import glob

train_imgs = glob.glob(f'{DATA_DIR}/images/train/*.jpg') + glob.glob(f'{DATA_DIR}/images/train/*.png')
val_imgs   = glob.glob(f'{DATA_DIR}/images/val/*.jpg')   + glob.glob(f'{DATA_DIR}/images/val/*.png')
train_lbls = glob.glob(f'{DATA_DIR}/labels/train/*.txt')
val_lbls   = glob.glob(f'{DATA_DIR}/labels/val/*.txt')

print(f'Train : {len(train_imgs):3d} ảnh | {len(train_lbls):3d} nhãn')
print(f'Val   : {len(val_imgs):3d} ảnh | {len(val_lbls):3d} nhãn')

if train_imgs and train_lbls:
    print('\n✅ Dataset hợp lệ — tiếp tục bước tiếp theo')
    sample = train_lbls[0]
    print(f'\nVí dụ nhãn ({os.path.basename(sample)}):')
    with open(sample) as f:
        for line in f.readlines()[:3]:
            parts = line.strip().split()
            cls_name = CLASSES[int(parts[0])] if int(parts[0]) < len(CLASSES) else '?'
            print(f'  class={parts[0]}({cls_name})  cx={parts[1]}  cy={parts[2]}  w={parts[3]}  h={parts[4]}')
else:
    print('\n❌ Chưa có dữ liệu — hãy upload dataset trước!')

---
## 🏷️ Bước 4: Tạo Label Map

TF Object Detection API dùng file `.pbtxt` để ánh xạ **ID lớp → tên lớp**.

**Quan trọng:** TF OD API đánh số bắt đầu từ **1** (YOLO bắt đầu từ 0).  
Việc chuyển đổi `YOLO class 0 → TF class 1` sẽ được xử lý tự động trong bước TFRecord.

In [ ]:
# Tạo label_map.pbtxt — định dạng protobuf text của TF OD API
with open(LABEL_MAP_PATH, 'w') as f:
    for i, name in enumerate(CLASSES, start=1):
        f.write(f'item {{\n  id: {i}\n  name: \'{name}\'\n}}\n\n')

print(f'✅ Đã tạo: {LABEL_MAP_PATH}')
print()
print(open(LABEL_MAP_PATH).read())

---
## 🔄 Bước 5: Convert YOLO → TFRecord

**TFRecord** là định dạng nhị phân tối ưu của TensorFlow — đọc nhanh hơn nhiều so với ảnh JPEG rời.

Quá trình chuyển đổi:
1. Đọc ảnh → encode thành bytes
2. Đọc file `.txt` YOLO → tính `xmin/xmax/ymin/ymax` từ `cx/cy/w/h`
3. Chuyển class ID từ 0-indexed → 1-indexed
4. Ghi vào file `.tfrecord`

In [ ]:
import io
import tensorflow as tf
from PIL import Image
from tqdm import tqdm

def yolo_to_tf_example(image_path, label_path, class_names):
    """Chuyển 1 ảnh + nhãn YOLO thành tf.train.Example."""
    with open(image_path, 'rb') as f:
        encoded = f.read()
    img = Image.open(io.BytesIO(encoded)).convert('RGB')
    w, h = img.size

    # Xác định format ảnh
    ext = os.path.splitext(image_path)[1].lower().lstrip('.')
    fmt = b'jpeg' if ext in ('jpg', 'jpeg') else ext.encode()
    fname = os.path.basename(image_path).encode('utf8')

    xmins, xmaxs, ymins, ymaxs = [], [], [], []
    class_texts, class_ids = [], []

    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) != 5:
                    continue
                # YOLO: class_id cx cy bw bh (tất cả normalized 0-1)
                cls = int(parts[0])
                cx, cy, bw, bh = map(float, parts[1:])
                # Chuyển từ center format → corner format
                xmins.append(max(0.0, cx - bw / 2))
                xmaxs.append(min(1.0, cx + bw / 2))
                ymins.append(max(0.0, cy - bh / 2))
                ymaxs.append(min(1.0, cy + bh / 2))
                class_texts.append(class_names[cls].encode('utf8'))
                class_ids.append(cls + 1)   # YOLO 0-index → TF OD API 1-index

    def b(v):  return tf.train.Feature(bytes_list=tf.train.BytesList(value=[v]))
    def i(v):  return tf.train.Feature(int64_list=tf.train.Int64List(value=[v]))
    def fl(v): return tf.train.Feature(float_list=tf.train.FloatList(value=v))
    def il(v): return tf.train.Feature(int64_list=tf.train.Int64List(value=v))
    def bl(v): return tf.train.Feature(bytes_list=tf.train.BytesList(value=v))

    feature = {
        'image/height':             i(h),
        'image/width':              i(w),
        'image/filename':           b(fname),
        'image/source_id':          b(fname),
        'image/encoded':            b(encoded),
        'image/format':             b(fmt),
        'image/object/bbox/xmin':   fl(xmins),
        'image/object/bbox/xmax':   fl(xmaxs),
        'image/object/bbox/ymin':   fl(ymins),
        'image/object/bbox/ymax':   fl(ymaxs),
        'image/object/class/text':  bl(class_texts),
        'image/object/class/label': il(class_ids),
    }
    return tf.train.Example(features=tf.train.Features(feature=feature))


def convert_split(split_name, class_names):
    img_dir = f'{DATA_DIR}/images/{split_name}'
    lbl_dir = f'{DATA_DIR}/labels/{split_name}'
    out_path = f'{TFRECORD_DIR}/{split_name}.tfrecord'

    imgs = sorted(
        glob.glob(f'{img_dir}/*.jpg') +
        glob.glob(f'{img_dir}/*.jpeg') +
        glob.glob(f'{img_dir}/*.png')
    )
    if not imgs:
        print(f'  ⚠️  Không có ảnh trong {img_dir}')
        return

    writer = tf.io.TFRecordWriter(out_path)
    ok = 0
    for img_path in tqdm(imgs, desc=f'  {split_name}'):
        stem = os.path.splitext(os.path.basename(img_path))[0]
        lbl_path = f'{lbl_dir}/{stem}.txt'
        try:
            ex = yolo_to_tf_example(img_path, lbl_path, class_names)
            writer.write(ex.SerializeToString())
            ok += 1
        except Exception as e:
            print(f'    Bỏ qua {os.path.basename(img_path)}: {e}')
    writer.close()
    size_mb = os.path.getsize(out_path) / 1024 / 1024
    print(f'  ✅ {ok}/{len(imgs)} ảnh → {out_path} ({size_mb:.1f} MB)')


print('Đang chuyển đổi YOLO → TFRecord...')
convert_split('train', CLASSES)
convert_split('val', CLASSES)
print('\n✅ TFRecord hoàn tất!')

---
## 🔽 Bước 6: Tải Pretrained Model

Tải checkpoint **SSD MobileNet V2 FPNLite 320×320** đã pretrain trên COCO 2017.

**Fine-tuning từ COCO** giúp:
- Model đã biết phát hiện hình chữ nhật, góc, khuôn mặt từ 80 lớp COCO
- Hội tụ nhanh trên dataset 462 ảnh (thay vì train from scratch cần hàng chục nghìn ảnh)
- Kết quả tốt hơn sau ít epochs hơn

In [ ]:
import tarfile, urllib.request

MODEL_URL = (
    'http://download.tensorflow.org/models/object_detection/tf2/20200711/'
    f'{MODEL_NAME}.tar.gz'
)
ARCHIVE   = f'{PRETRAINED_DIR}/{MODEL_NAME}.tar.gz'
CKPT_PATH = f'{PRETRAINED_DIR}/{MODEL_NAME}/checkpoint/ckpt-0.index'

if os.path.isfile(CKPT_PATH):
    print(f'✅ Đã tải trước đó: {PRETRAINED_CKPT}')
else:
    def _progress(b, bs, total):
        print(f'\r  {min(b*bs/total*100, 100):.1f}%', end='', flush=True)

    print(f'Đang tải {MODEL_NAME} (~22 MB)...')
    urllib.request.urlretrieve(MODEL_URL, ARCHIVE, reporthook=_progress)
    print('\nĐang giải nén...')
    with tarfile.open(ARCHIVE) as t:
        t.extractall(PRETRAINED_DIR)
    os.remove(ARCHIVE)
    print(f'✅ Đã lưu checkpoint: {PRETRAINED_CKPT}')

# Hiển thị các file checkpoint
ckpt_dir = os.path.dirname(PRETRAINED_CKPT)
print('\nFiles trong checkpoint:')
for f in os.listdir(ckpt_dir):
    print(f'  {f}')

---
## ⚙️ Bước 7: Tạo Pipeline Config

`pipeline.config` là file điều khiển **toàn bộ quá trình training** của TF OD API, bao gồm:

| Phần | Nội dung |
|------|----------|
| `model` | Kiến trúc SSD + backbone MobileNetV2 + FPN head |
| `train_config` | Optimizer, learning rate schedule, augmentation |
| `train_input_reader` | Đường dẫn TFRecord train + label map |
| `eval_input_reader` | Đường dẫn TFRecord validation |

**Learning rate:** Cosine decay với warmup — phù hợp fine-tuning trên dataset nhỏ.

In [ ]:
# Sinh pipeline.config theo giá trị đã đặt ở Bước 2
# Dùng cách viết từng dòng để tránh xung đột { } với Python format string

TRAIN_RECORD = f'{TFRECORD_DIR}/train.tfrecord'
VAL_RECORD   = f'{TFRECORD_DIR}/val.tfrecord'

lines = [
    '# SSD MobileNet V2 FPNLite 320x320 — eKYC CCCD Detection',
    'model {',
    '  ssd {',
    f'    num_classes: {NUM_CLASSES}',
    '    image_resizer {',
    '      fixed_shape_resizer {',
    f'        height: {IMAGE_SIZE}',
    f'        width: {IMAGE_SIZE}',
    '      }',
    '    }',
    '    feature_extractor {',
    '      type: "ssd_mobilenet_v2_fpn_keras"',
    '      depth_multiplier: 1.0',
    '      min_depth: 16',
    '      conv_hyperparams {',
    '        activation: RELU_6',
    '        regularizer { l2_regularizer { weight: 0.00004 } }',
    '        initializer { random_truncated_normal_initializer { stddev: 0.03 mean: 0.0 } }',
    '        batch_norm { decay: 0.997 center: true scale: true epsilon: 0.001 train: true }',
    '      }',
    '      use_depthwise: true',
    '      override_base_feature_extractor_hyperparams: true',
    '      fpn { min_level: 3 max_level: 7 additional_layer_depth: 128 }',
    '    }',
    '    box_coder {',
    '      faster_rcnn_box_coder {',
    '        y_scale: 10.0  x_scale: 10.0  height_scale: 5.0  width_scale: 5.0',
    '      }',
    '    }',
    '    matcher {',
    '      argmax_matcher {',
    '        matched_threshold: 0.5  unmatched_threshold: 0.5',
    '        ignore_thresholds: false  negatives_lower_than_unmatched: true',
    '        force_match_for_each_row: true  use_matmul_gather: true',
    '      }',
    '    }',
    '    similarity_calculator { iou_similarity {} }',
    '    box_predictor {',
    '      weight_shared_convolutional_box_predictor {',
    '        depth: 128',
    '        class_prediction_bias_init: -4.6',
    '        conv_hyperparams {',
    '          activation: RELU_6',
    '          regularizer { l2_regularizer { weight: 0.00004 } }',
    '          initializer { random_truncated_normal_initializer { stddev: 0.03 mean: 0.0 } }',
    '          batch_norm { decay: 0.997 center: true scale: true epsilon: 0.001 train: true }',
    '        }',
    '        num_layers_before_predictor: 4',
    '        kernel_size: 3',
    '        use_depthwise: true',
    '      }',
    '    }',
    '    anchor_generator {',
    '      multiscale_anchor_generator {',
    '        min_level: 3  max_level: 7  anchor_scale: 4.0',
    '        aspect_ratios: [1.0, 2.0, 0.5]  scales_per_octave: 2',
    '      }',
    '    }',
    '    post_processing {',
    '      batch_non_max_suppression {',
    '        score_threshold: 1e-8  iou_threshold: 0.6',
    '        max_detections_per_class: 100  max_total_detections: 100',
    '      }',
    '      score_converter: SIGMOID',
    '    }',
    '    normalize_loss_by_num_matches: true',
    '    loss {',
    '      localization_loss { weighted_smooth_l1 { delta: 1.0 } }',
    '      classification_loss { weighted_sigmoid_focal { gamma: 2.0 alpha: 0.25 } }',
    '      classification_weight: 1.0  localization_weight: 1.0',
    '    }',
    '    encode_background_as_zeros: true',
    '    normalize_loc_loss_by_codesize: true',
    '    inplace_batchnorm_update: true',
    '    freeze_batchnorm: false',
    '  }',
    '}',
    '',
    'train_config {',
    f'  batch_size: {BATCH_SIZE}',
    '  data_augmentation_options { random_horizontal_flip {} }',
    '  data_augmentation_options {',
    '    random_crop_image {',
    '      min_object_covered: 0.0  min_aspect_ratio: 0.75  max_aspect_ratio: 3.0',
    '      min_area: 0.75  max_area: 1.0  overlap_thresh: 0.0',
    '    }',
    '  }',
    '  sync_replicas: true',
    '  optimizer {',
    '    momentum_optimizer {',
    '      learning_rate {',
    '        cosine_decay_learning_rate {',
    f'          learning_rate_base: {LR_BASE}',
    f'          total_steps: {NUM_STEPS}',
    f'          warmup_learning_rate: {WARMUP_LR}',
    f'          warmup_steps: {WARMUP_STEPS}',
    '        }',
    '      }',
    '      momentum_optimizer_value: 0.9',
    '    }',
    '    use_moving_average: false',
    '  }',
    f'  fine_tune_checkpoint: "{PRETRAINED_CKPT}"',
    '  fine_tune_checkpoint_type: "detection"',
    '  fine_tune_checkpoint_version: V2',
    f'  num_steps: {NUM_STEPS}',
    '  startup_delay_steps: 0.0',
    '  replicas_to_aggregate: 8',
    '  max_number_of_boxes: 100',
    '  unpad_groundtruth_tensors: false',
    '}',
    '',
    'train_input_reader {',
    f'  label_map_path: "{LABEL_MAP_PATH}"',
    '  tf_record_input_reader {',
    f'    input_path: "{TRAIN_RECORD}"',
    '  }',
    '}',
    '',
    'eval_config {',
    '  metrics_set: "coco_detection_metrics"',
    '  use_moving_averages: false',
    '}',
    '',
    'eval_input_reader {',
    f'  label_map_path: "{LABEL_MAP_PATH}"',
    '  shuffle: false',
    '  num_epochs: 1',
    '  tf_record_input_reader {',
    f'    input_path: "{VAL_RECORD}"',
    '  }',
    '}',
]

with open(CONFIG_PATH, 'w') as f:
    f.write('\n'.join(lines))

print(f'✅ pipeline.config đã tạo: {CONFIG_PATH}')
print(f'   num_classes  : {NUM_CLASSES}')
print(f'   image_size   : {IMAGE_SIZE}×{IMAGE_SIZE}')
print(f'   batch_size   : {BATCH_SIZE}')
print(f'   num_steps    : {NUM_STEPS:,}')
print(f'   fine_tune_from: {PRETRAINED_CKPT}')

---
## 🏋️ Bước 8: Huấn luyện Model

Training dùng `model_main_tf2.py` từ TF OD API — script này xử lý:
- Load pipeline config
- Build SSD MobileNetV2 model
- Fine-tune từ COCO checkpoint
- Tự động eval trên validation set mỗi checkpoint
- Log metrics cho TensorBoard

> 💡 **Theo dõi loss:** Mở TensorBoard ở Bước 9 trong một tab riêng  
> 💡 **Dừng sớm:** Ctrl+C — checkpoint cuối cùng vẫn được giữ lại

In [ ]:
import sys, importlib.util

# Tất cả biến môi trường cần truyền vào subprocess (model_main_tf2.py, exporter_main_v2.py)
# Subprocess tạo ra bởi !python chạy trong môi trường riêng — không kế thừa os.environ
# của kernel Jupyter, nhưng THỪA KẾ biến đặt trước lệnh !python ... trên bash
OD_PYTHONPATH = '/content/tf_models/research:/content/tf_models/research/slim'
OD_ENV = (
    f'PYTHONPATH={OD_PYTHONPATH} '
    f'PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION=python '
    f'TF_USE_LEGACY_KERAS=1'         # ← fix "layers has no attribute experimental"
)

# Thêm vào kernel path
for p in OD_PYTHONPATH.split(':'):
    if p not in sys.path:
        sys.path.insert(0, p)


def find_od_script(name):
    """Tìm script TF OD API — ưu tiên repo đã clone."""
    cloned = f'/content/tf_models/research/object_detection/{name}'
    if os.path.isfile(cloned):
        return cloned
    spec = importlib.util.find_spec('object_detection')
    if spec is None:
        raise RuntimeError('object_detection không tìm thấy — chạy lại cell cài đặt.')
    pkg = os.path.dirname(spec.origin)
    p = os.path.join(pkg, name)
    if not os.path.isfile(p):
        p = os.path.join(os.path.dirname(pkg), name)
    if not os.path.isfile(p):
        raise RuntimeError(f'{name} không tìm thấy gần {pkg}')
    return p


train_script = find_od_script('model_main_tf2.py')
print(f'Script : {train_script}')
print(f'Config : {CONFIG_PATH}')
print(f'Output : {CKPT_DIR}')
print(f'Steps  : {NUM_STEPS:,}')
print(f'Env    : {OD_ENV}')
print()
print('San sang. Chay cell tiep theo de bat dau training.')

In [ ]:
# Chạy training với đầy đủ biến môi trường cho subprocess:
#   PYTHONPATH                               → tìm object_detection
#   PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION  → tránh lỗi _pb2.py TypeError

!{OD_ENV} python {train_script} \
    --pipeline_config_path={CONFIG_PATH} \
    --model_dir={CKPT_DIR} \
    --num_train_steps={NUM_STEPS} \
    --alsologtostderr

In [ ]:
# Kiểm tra checkpoint đã lưu sau khi training
ckpts = sorted(glob.glob(f'{CKPT_DIR}/ckpt-*.index'))
if ckpts:
    print(f'✅ Tìm thấy {len(ckpts)} checkpoint:')
    for c in ckpts[-5:]:  # hiển thị 5 checkpoint gần nhất
        print(f'  {os.path.basename(c)}')
else:
    print('⚠️  Chưa có checkpoint — training chưa chạy hoặc bị lỗi')

---
## 📊 Bước 9: Theo dõi với TensorBoard

TensorBoard hiển thị **loss** và **mAP** theo từng bước training.  
Chạy cell này **song song** với cell training (mở tab mới) hoặc sau khi xong.

In [ ]:
# Load TensorBoard extension trong Colab
%load_ext tensorboard
%tensorboard --logdir {CKPT_DIR}

---
## 💾 Bước 10: Export sang SavedModel

Chuyển checkpoint training → **SavedModel** (định dạng chuẩn TF2 cho serving).  
SavedModel là bước trung gian trước khi convert sang TFLite.

**Input type:** `image_tensor` — nhận vào batch ảnh `[N, H, W, 3]` uint8

In [ ]:
export_script = find_od_script('exporter_main_v2.py')
print(f'Exporter: {export_script}')

!{OD_ENV} python {export_script} \
    --input_type=image_tensor \
    --pipeline_config_path={CONFIG_PATH} \
    --trained_checkpoint_dir={CKPT_DIR} \
    --output_directory={EXPORT_DIR}

In [ ]:
SAVED_MODEL_DIR = f'{EXPORT_DIR}/saved_model'

if os.path.isdir(SAVED_MODEL_DIR):
    print(f'✅ SavedModel: {SAVED_MODEL_DIR}')
    total = sum(os.path.getsize(os.path.join(r, f))
                for r, _, fs in os.walk(SAVED_MODEL_DIR) for f in fs)
    print(f'   Kích thước : {total/1024/1024:.1f} MB')

    # Thử load model để xác nhận
    import tensorflow as tf
    model = tf.saved_model.load(SAVED_MODEL_DIR)
    print('   Load test  : ✅ OK')
else:
    print(f'❌ SavedModel chưa tạo — chạy lại cell export')

---
## 📱 Bước 11: Chuyển đổi sang TFLite

Tạo **3 phiên bản TFLite** với các mức quantization khác nhau:

| File | Phương pháp | Kích thước | Dùng khi |
|------|-------------|-----------|----------|
| `model_fp32.tflite` | Không quantize | ~22 MB | Debug, server |
| `model_fp16.tflite` | Float16 weights | ~11 MB | Android GPU delegate |
| `model_int8.tflite` | Full int8 | ~6 MB | Mobile CPU, Raspberry Pi |

> 🎯 **Khuyến nghị cho eKYC mobile:** dùng `model_fp16.tflite` — cân bằng tốt giữa kích thước và độ chính xác

In [ ]:
import tensorflow as tf
import numpy as np
from PIL import Image as PILImage

# --- fp32: không quantize (baseline) ---
print('[1/3] Đang tạo model_fp32.tflite...')
converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)
tflite_fp32 = converter.convert()
fp32_path = f'{TFLITE_DIR}/model_fp32.tflite'
with open(fp32_path, 'wb') as f:
    f.write(tflite_fp32)
print(f'  ✅ {fp32_path} ({os.path.getsize(fp32_path)/1024/1024:.1f} MB)')

# --- fp16: chỉ quantize weights sang float16 ---
print('\n[2/3] Đang tạo model_fp16.tflite...')
converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)
converter.optimizations = [tf.lite.Optimize.DEFAULT]           # bật quantize
converter.target_spec.supported_types = [tf.float16]           # target float16
tflite_fp16 = converter.convert()
fp16_path = f'{TFLITE_DIR}/model_fp16.tflite'
with open(fp16_path, 'wb') as f:
    f.write(tflite_fp16)
print(f'  ✅ {fp16_path} ({os.path.getsize(fp16_path)/1024/1024:.1f} MB)')

In [ ]:
# --- int8: full integer quantization (cần ảnh hiệu chỉnh) ---
print('[3/3] Đang tạo model_int8.tflite (cần ~2-5 phút)...')

# Lấy ảnh hiệu chỉnh từ tập train (100 ảnh là đủ)
calib_imgs = (glob.glob(f'{DATA_DIR}/images/train/*.jpg') +
              glob.glob(f'{DATA_DIR}/images/train/*.png'))[:100]

def representative_dataset_gen():
    """Cung cấp ảnh thực để TFLite calibrate dải giá trị activations."""
    for path in calib_imgs:
        img = PILImage.open(path).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
        arr = np.array(img, dtype=np.float32)[np.newaxis]   # [1, 320, 320, 3]
        yield [arr]

converter = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL_DIR)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset_gen
# Yêu cầu tất cả ops chạy trên int8; fallback sang float cho ops không hỗ trợ
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8,
    tf.lite.OpsSet.TFLITE_BUILTINS,
]
converter.inference_input_type  = tf.uint8    # input là ảnh uint8 [0,255]
converter.inference_output_type = tf.float32  # giữ output float để dễ xử lý

tflite_int8 = converter.convert()
int8_path = f'{TFLITE_DIR}/model_int8.tflite'
with open(int8_path, 'wb') as f:
    f.write(tflite_int8)
print(f'  ✅ {int8_path} ({os.path.getsize(int8_path)/1024/1024:.1f} MB)')

In [ ]:
# So sánh kích thước các model
import matplotlib.pyplot as plt

models = {
    'SavedModel': sum(os.path.getsize(os.path.join(r,f))
                     for r,_,fs in os.walk(SAVED_MODEL_DIR) for f in fs),
    'TFLite fp32': os.path.getsize(fp32_path),
    'TFLite fp16': os.path.getsize(fp16_path),
    'TFLite int8': os.path.getsize(int8_path),
}

print('\n📊 So sánh kích thước model:')
print(f'{"Model":<18} {"Kích thước":>12}')
print('-' * 32)
for name, size in models.items():
    bar = '█' * int(size / models['SavedModel'] * 20)
    print(f'{name:<18} {size/1024/1024:>8.1f} MB  {bar}')

fp32_mb = models['TFLite fp32'] / 1024 / 1024
print(f'\n  fp16 nhỏ hơn fp32: {fp32_mb/models["TFLite fp16"]*1024/1024:.1f}×')
print(f'  int8 nhỏ hơn fp32: {fp32_mb/models["TFLite int8"]*1024/1024:.1f}×')

---
## 🔍 Bước 12: Test Inference trên ảnh

Chạy model TFLite trực tiếp — **không cần TF OD API**, chỉ cần `tensorflow` hoặc `tflite-runtime`.
Đây là code bạn sẽ tích hợp vào app mobile/web.

In [ ]:
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image as PILImage

# Tên lớp: index 0 là background (TF OD API), 1-5 là các lớp thực
CLASS_NAMES = ['background'] + CLASSES
COLORS = ['#00FF00', '#FF0000', '#0000FF', '#FFA500', '#800080']


def run_tflite_inference(model_path, image_path, score_threshold=0.4):
    """Chạy inference TFLite và vẽ kết quả."""

    # 1. Load interpreter
    interp = tf.lite.Interpreter(model_path=model_path, num_threads=4)
    interp.allocate_tensors()
    in_detail  = interp.get_input_details()[0]
    out_details = {d['name']: d for d in interp.get_output_details()}

    # 2. Chuẩn bị ảnh đầu vào
    img_orig = PILImage.open(image_path).convert('RGB')
    orig_w, orig_h = img_orig.size
    img_resized = img_orig.resize((IMAGE_SIZE, IMAGE_SIZE))
    input_arr = np.array(img_resized)[np.newaxis]  # [1, 320, 320, 3]

    if in_detail['dtype'] == np.float32:
        input_arr = input_arr.astype(np.float32)
    else:
        input_arr = input_arr.astype(np.uint8)   # int8 model dùng uint8 input

    # 3. Chạy inference
    import time
    interp.set_tensor(in_detail['index'], input_arr)
    t0 = time.perf_counter()
    interp.invoke()
    latency = (time.perf_counter() - t0) * 1000

    # 4. Lấy kết quả
    def get_out(key_fragment):
        for name, d in out_details.items():
            if key_fragment in name:
                return interp.get_tensor(d['index'])
        return list(out_details.values())[0]  # fallback

    boxes   = get_out('boxes')[0]       # [N, 4] — ymin xmin ymax xmax (0-1)
    scores  = get_out('scores')[0]      # [N]
    classes = get_out('classes')[0].astype(int)  # [N]

    # 5. Lọc theo threshold và vẽ
    fig, ax = plt.subplots(1, figsize=(10, 8))
    ax.imshow(img_orig)

    detected = []
    for box, score, cls in zip(boxes, scores, classes):
        if score < score_threshold:
            continue
        ymin, xmin, ymax, xmax = box
        x = xmin * orig_w;  y = ymin * orig_h
        bw = (xmax - xmin) * orig_w;  bh = (ymax - ymin) * orig_h
        color = COLORS[(cls - 1) % len(COLORS)]
        rect = patches.Rectangle((x, y), bw, bh,
                                   linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        label = f'{CLASS_NAMES[cls]}: {score:.2f}'
        ax.text(x, y - 5, label, color=color, fontsize=9,
                bbox=dict(facecolor='black', alpha=0.5, pad=2))
        detected.append((CLASS_NAMES[cls], score))

    model_name = os.path.basename(model_path)
    ax.set_title(f'{model_name} | latency={latency:.1f}ms | detected={len(detected)}')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

    print(f'\nLatency: {latency:.1f} ms')
    for cls_name, score in detected:
        print(f'  {cls_name:<18} score={score:.3f}')
    if not detected:
        print('  (không phát hiện đối tượng nào — thử giảm score_threshold)')
    return detected


print('Hàm run_tflite_inference đã sẵn sàng')
print('Chạy cell tiếp theo để test inference')

In [ ]:
# Chọn ảnh test — thay bằng đường dẫn ảnh của bạn
test_images = (
    glob.glob(f'{DATA_DIR}/images/val/*.jpg') +
    glob.glob(f'{DATA_DIR}/images/val/*.png')
)

if not test_images:
    print('⚠️  Không có ảnh val — thử dùng ảnh train hoặc upload ảnh mới')
else:
    test_img = test_images[0]   # ← thay chỉ số để chọn ảnh khác
    print(f'Ảnh test: {test_img}')

    # Test với model fp16 (khuyến nghị)
    print('\n--- fp16 ---')
    run_tflite_inference(fp16_path, test_img, score_threshold=0.4)

    # Test với model int8
    print('\n--- int8 ---')
    run_tflite_inference(int8_path, test_img, score_threshold=0.4)

In [ ]:
# Upload ảnh từ máy tính để test nhanh
from google.colab import files

print('Upload ảnh CCCD để test...')
uploaded_test = files.upload()

for fname in uploaded_test:
    img_path = f'/tmp/{fname}'
    with open(img_path, 'wb') as f:
        f.write(uploaded_test[fname])
    print(f'\nTest với: {fname}')
    run_tflite_inference(fp16_path, img_path, score_threshold=0.3)

---
## 💾 Bước 13: Lưu model về Google Drive & Download

Lưu kết quả để không mất khi Colab session hết hạn.

In [ ]:
import shutil

if SAVE_TO_DRIVE:
    # Tạo thư mục lưu trên Drive
    drive_tflite = f'{DRIVE_SAVE_DIR}/tflite'
    drive_saved  = f'{DRIVE_SAVE_DIR}/saved_model'
    os.makedirs(drive_tflite, exist_ok=True)
    os.makedirs(drive_saved, exist_ok=True)

    # Copy TFLite models
    for fname in ['model_fp32.tflite', 'model_fp16.tflite', 'model_int8.tflite']:
        src = f'{TFLITE_DIR}/{fname}'
        dst = f'{drive_tflite}/{fname}'
        if os.path.isfile(src):
            shutil.copy2(src, dst)
            print(f'✅ Đã lưu → {dst}')

    # Copy label map
    shutil.copy2(LABEL_MAP_PATH, f'{DRIVE_SAVE_DIR}/label_map.pbtxt')
    print(f'✅ Đã lưu label_map.pbtxt → {DRIVE_SAVE_DIR}')

    # Copy SavedModel (lớn hơn)
    if os.path.isdir(SAVED_MODEL_DIR):
        shutil.copytree(SAVED_MODEL_DIR, drive_saved, dirs_exist_ok=True)
        print(f'✅ Đã lưu SavedModel → {drive_saved}')

    print(f'\n📁 Tất cả model đã lưu tại: {DRIVE_SAVE_DIR}')
else:
    print('SAVE_TO_DRIVE = False — bỏ qua bước lưu Drive')

In [ ]:
# Download model về máy tính
from google.colab import files
import zipfile

# Nén tất cả TFLite model + label map vào 1 file ZIP
zip_path = '/tmp/ekyc_cccd_tflite_models.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in ['model_fp16.tflite', 'model_int8.tflite', 'model_fp32.tflite']:
        src = f'{TFLITE_DIR}/{fname}'
        if os.path.isfile(src):
            zf.write(src, fname)
            print(f'  + {fname}')
    zf.write(LABEL_MAP_PATH, 'label_map.pbtxt')
    print('  + label_map.pbtxt')

total_mb = os.path.getsize(zip_path) / 1024 / 1024
print(f'\nZIP: {zip_path} ({total_mb:.1f} MB)')
print('Đang tải xuống...')
files.download(zip_path)

---
## ✅ Tổng kết Pipeline

```
YOLO Dataset (462 ảnh)
       │
       ▼  Bước 5: Convert
   TFRecord (train.tfrecord + val.tfrecord)
       │
       ▼  Bước 8: Fine-tune 10,000 steps
   Checkpoint (ckpt-*.index)
       │
       ▼  Bước 10: Export
   SavedModel (~22 MB)
       │
       ├──▶  model_fp32.tflite  (~22 MB) — server/debug
       ├──▶  model_fp16.tflite  (~11 MB) — mobile GPU ✅ khuyến nghị
       └──▶  model_int8.tflite  (~ 6 MB) — mobile CPU tối ưu
```

### Bước tiếp theo
- **Android**: dùng [TFLite Task Library](https://www.tensorflow.org/lite/inference_with_metadata/task_library/object_detector) với `model_fp16.tflite`
- **iOS**: dùng TFLite Swift/ObjC API
- **Server Python**: load với `tf.lite.Interpreter` (xem code inference Bước 12)
- **Cải thiện accuracy**: tăng `NUM_STEPS` lên 20,000 hoặc 50,000; thêm augmentation